<a href="https://colab.research.google.com/github/rlpinhal/Aprendendo_Python/blob/main/aula_16_ApiTesteExcel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile app.py
import requests
import json

API_URL = 'https://script.google.com/macros/s/AKfycbzT_McJiGQXi27acijGm08B7-eYe4fs7TF-jFEFO9Ac2PbAthHCxH4KimmCTxz91ntigg/exec'  # Replace with your actual Google Apps Script web app URL

def send_message(name, email):
    response = requests.post(
        API_URL,
        json={"name": name, "email": email}
    )
    print(response.text)

def send_multiple_messages(messages):
    for message in messages:
        send_message(message['name'], message['email'])

def get_all_messages():
    response = requests.get(API_URL)
    data = response.json()
    print(json.dumps(data, indent=2))

def get_last_message():
    response = requests.get(API_URL)
    data = response.json()
    if data:
        last = data[-1]
        print(json.dumps(last, indent=2))
    else:
        print("No messages found.")

if __name__ == "__main__":
    print("Sending one message...")
    send_message("alice", "alice@example.com")

    print("\nSending multiple messages...")
    messages = [
        {"name": "bob", "email": "bob@example.com"},
        {"name": "charlie", "email": "charlie@example.com"}
    ]
    send_multiple_messages(messages)

    print("\nRetrieving all messages...")
    get_all_messages()

    print("\nRetrieving last message...")
    get_last_message()



Overwriting app.py


In [ ]:
%%writefile test.py
import pytest
import json
from io import StringIO
import sys
import time
import os

sys.path.append(os.path.abspath(os.getcwd()))

from app import send_message, send_multiple_messages, get_all_messages, get_last_message, API_URL

def capture_print(func, *args, **kwargs):
    """Utility to capture print output of a function."""
    old_stdout = sys.stdout
    sys.stdout = mystdout = StringIO()
    try:
        func(*args, **kwargs)
    finally:
        sys.stdout = old_stdout
    return mystdout.getvalue()

@pytest.fixture(scope="module")
def seeded_sheet():
    # Clean state is not guaranteed, but let's try to send the 3 messages
    capture_print(send_message, "alice", "alice@example.com")
    time.sleep(1)
    capture_print(send_multiple_messages, [
        {"name": "bob", "email": "bob@example.com"},
        {"name": "charlie", "email": "charlie@example.com"}
    ])
    time.sleep(2)  # Give Sheets a moment to update
    return True

def test_send_message(seeded_sheet):
    output = capture_print(send_message, "testuser", "testuser@example.com")
    assert "Success" in output or "success" in output.lower()

def test_send_multiple_messages(seeded_sheet):
    output = capture_print(send_multiple_messages, [
        {"name": "testuser2", "email": "testuser2@example.com"},
        {"name": "testuser3", "email": "testuser3@example.com"}
    ])
    # Should print success twice
    assert output.lower().count("success") == 2

def test_get_all_messages(seeded_sheet):
    # get_all_messages prints the JSON array
    output = capture_print(get_all_messages)
    data = json.loads(output)
    # There should be at least our test entries present
    assert isinstance(data, list)
    assert any(e["name"] == "alice" and e["email"] == "alice@example.com" for e in data)
    assert any(e["name"] == "bob" and e["email"] == "bob@example.com" for e in data)
    assert any(e["name"] == "charlie" and e["email"] == "charlie@example.com" for e in data)
    assert any(e["name"] == "testuser2" and e["email"] == "testuser2@example.com" for e in data)
    assert any(e["name"] == "testuser3" and e["email"] == "testuser3@example.com" for e in data)

def test_get_last_message(seeded_sheet):
    output = capture_print(get_last_message)
    last_data = json.loads(output)
    assert last_data["name"] == "testuser3"
    assert last_data["email"] == "testuser3@example.com"


Overwriting test.py


In [ ]:
!pytest -v test.py


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.7.30, anyio-4.13.0, typeguard-4.5.1
collected 4 items                                                              

test.py::test_send_message PASSED                                        [ 25%]
test.py::test_send_multiple_messages PASSED                              [ 50%]
test.py::test_get_all_messages PASSED                                    [ 75%]
test.py::test_get_last_message PASSED                                    [100%]

============================== 4 passed in 39.59s ==============================
